# PantheraLAB 🐆 · Laboratorio de detección real v3

**Objetivo:** mejorar la localización del animal y ofrecer resultados visuales útiles a investigadores, conservando la fotografía **completa**, las cajas **detectadas de verdad** y los recortes que utiliza SpeciesNet.

**Qué cambió:** probamos MegaDetector V6 Extra con la foto original y dos orientaciones (0°, 90°, 270°), regresamos las cajas a las coordenadas originales y comparamos SpeciesNet con el recorte exacto y otro con contexto. El segundo recorte **es una ampliación para clasificar**, NO una caja detectada por la IA.

**Probado previamente:** con tu perro, la detección original fue 78,6 %, pero la caja cubría principalmente la cabeza. SpeciesNet sugirió «Canis species» (56,02 %) y «domestic dog» (42,31 %). Son **puntuaciones de modelos**, no exactitud científica ni identificación verificada.

1. Abrí **Entorno de ejecución → Ejecutar todas**.
2. Subí una foto propia del animal. La misma fotografía sirve para comparar.
3. Revisá los paneles de la **foto completa**, las cajas del modelo y los recortes utilizados. Descargá el informe al final.

Este laboratorio procesa **fotografías en una sesión temporal de Google Colab**; no analiza video continuo ni envía nada a PantheraLIVE. Evitá imágenes con personas o información sensible. Las orientaciones múltiples pueden tardar varios minutos, especialmente en CPU.

Repositorios oficiales: https://github.com/microsoft/MegaDetector · https://github.com/google/cameratrapai

## 1. Instalar MegaDetector y SpeciesNet
Solo en la sesión de Colab. Puede tardar varios minutos en descargar paquetes.

In [ ]:
%pip -q install PytorchWildlife speciesnet Pillow

## 2. Elegir la foto del perro u otro animal
Se ejecuta una única prueba sobre tu fotografía. Recomendación: JPG, PNG o WEBP de hasta 12 MB.

In [ ]:
from google.colab import files
from IPython.display import display
from PIL import Image, ImageOps
from pathlib import Path
from io import BytesIO
uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No se seleccionó una fotografía.")
name, payload = next(iter(uploaded.items()))
if len(payload)>12*1024*1024:
    raise ValueError("Fotografía mayor que 12 MB.")
photo = ImageOps.exif_transpose(Image.open(BytesIO(payload))).convert("RGB")
if photo.width*photo.height>25_000_000:
    raise ValueError("Fotografía mayor que 25 megapíxeles.")
folder = Path("/content/pantheralab_prueba_real")
folder.mkdir(parents=True,exist_ok=True)
file_path = folder/"fotograma.jpg"
photo.save(file_path,quality=92)
preview = photo.copy()
preview.thumbnail((820,820))
display(preview)
print("Fotografía preparada:",photo.width,"x",photo.height)

## 3. Detección multi-orientación · MegaDetector V6 Extra

El perro aparece acostado. Por eso ejecutamos el **mismo detector real** sobre la imagen original y sus versiones giradas 90° y 270°. Convertimos cada caja al sistema de coordenadas de la fotografía original.

Mostramos la **caja principal del modelo** en verde y otras propuestas en ámbar. La elección prioriza una caja de mayor cobertura **solo entre opciones de confianza parecida**; esto es una heurística experimental, no una validación de que encierre todo el cuerpo.

El umbral de 0,20 permite más candidatos, incluidos falsos positivos. Si una orientación no encuentra el cuerpo completo, el notebook lo indicará sin inventar la región.

In [ ]:
import torch
from PIL import Image, ImageDraw
from PytorchWildlife.models import detection as pw_detection

device="cuda" if torch.cuda.is_available() else "cpu"
variant="MDV6-yolov10-e"
threshold=0.20
orientations=[0,90,270]
print("Detector:",variant,"| Procesador:",device)
model=pw_detection.MegaDetectorV6(device=device,pretrained=True,version=variant)
names={0:"animal",1:"person",2:"vehicle"}
W,H=photo.size

def original_coordinates(rotated_box,angle):
    # Las cajas usan coordenadas de bordes, no índices de píxeles.
    x1,y1,x2,y2=[float(v) for v in rotated_box]
    corners=((x1,y1),(x1,y2),(x2,y1),(x2,y2))
    if angle==90:
        mapped=[(W-y,x) for x,y in corners]
    elif angle==270:
        mapped=[(y,H-x) for x,y in corners]
    else:
        mapped=list(corners)
    xa=max(0,min(W,min(x for x,y in mapped)))
    ya=max(0,min(H,min(y for x,y in mapped)))
    xb=max(0,min(W,max(x for x,y in mapped)))
    yb=max(0,min(H,max(y for x,y in mapped)))
    return [int(round(v)) for v in [xa,ya,xb,yb]]

def overlap(a,b):
    ax1,ay1,ax2,ay2=a["bbox_xyxy"]
    bx1,by1,bx2,by2=b["bbox_xyxy"]
    inter=max(0,min(ax2,bx2)-max(ax1,bx1))*max(0,min(ay2,by2)-max(ay1,by1))
    area_a=max(0,ax2-ax1)*max(0,ay2-ay1)
    area_b=max(0,bx2-bx1)*max(0,by2-by1)
    union=area_a+area_b-inter
    return inter/union if union else 0.0

raw_detections=[]
for angle in orientations:
    rotated=photo if angle==0 else photo.transpose(
        Image.Transpose.ROTATE_90 if angle==90 else Image.Transpose.ROTATE_270)
    temp_path=folder/("orientacion_"+str(angle)+".jpg")
    if angle==0:
        temp_path=file_path
    else:
        rotated.save(temp_path,quality=92)
    prediction=model.single_image_detection(str(temp_path),det_conf_thres=threshold)
    objects=prediction.get("detections")
    candidates=0
    if objects is not None:
        for box,confidence,category in zip(objects.xyxy,objects.confidence,objects.class_id):
            coords=original_coordinates(box,angle)
            x1,y1,x2,y2=coords
            area=max(0,x2-x1)*max(0,y2-y1)
            if area<=0:continue
            raw_detections.append({
                "class_name":names.get(int(category),"unknown"),
                "confidence":round(float(confidence),4),
                "bbox_xyxy":coords,
                "orientation_degrees":angle,
                "box_fraction_of_image":round(area/(W*H),5)
            })
            candidates+=1
    print("Orientación",angle,"°:",candidates,"candidatos")

# Quitar duplicados que se superponen mucho (NMS por clase).
detections=[]
for candidate in sorted(raw_detections,key=lambda d:d["confidence"],reverse=True):
    if any(d["class_name"]==candidate["class_name"]
           and overlap(d,candidate)>=0.65 for d in detections):
        continue
    detections.append(candidate)
detections=detections[:20]

animals=[d for d in detections if d["class_name"]=="animal"]
people=[d for d in detections if d["class_name"]=="person"]
primary_detection=None
if animals:
    highest=max(d["confidence"] for d in animals)
    comparable=[d for d in animals if d["confidence"]>=max(threshold,highest-0.15)]
    # Heurística: entre cajas con confianza comparable, preferir cobertura.
    primary_detection=max(comparable,key=lambda d:(d["box_fraction_of_image"],d["confidence"]))
    print("Caja principal: orientación",primary_detection["orientation_degrees"],
          "° | confianza",round(primary_detection["confidence"]*100,2),
          "% | cobertura",round(primary_detection["box_fraction_of_image"]*100,2),"% de la imagen")

annotated=photo.copy()
drawer=ImageDraw.Draw(annotated)
for item in detections:
    box=item["bbox_xyxy"]
    color="#00dc9c" if item is primary_detection else "#ffc36b"
    drawer.rectangle(box,outline=color,width=5 if item is primary_detection else 3)
    tag=item["class_name"].upper()+" "+str(round(item["confidence"]*100))+"% ("+str(item["orientation_degrees"])+"°)"
    drawer.text((box[0]+3,max(0,box[1]-18)),tag,fill=color)
annotated.save(folder/"detecciones.jpg",quality=92)
print("Animales:",len(animals),"| Personas:",len(people),
      "| Vehículos:",sum(x["class_name"]=="vehicle" for x in detections))
if not animals:
    print("⚠ El modelo no encontró un animal con el umbral actual.")
elif primary_detection["box_fraction_of_image"]<0.01:
    print("⚠ Caja muy pequeña para clasificar con fiabilidad.")
print("A continuación se mostrarán la foto completa y las regiones de la IA.")

## 4. Clasificación con SpeciesNet · comparación de recortes

SpeciesNet compara dos **entradas distintas del mismo animal**:

- **Recorte exacto:** la caja que produjo MegaDetector V6.
- **Recorte con contexto:** añadimos margen del 65 % de la caja para incluir pelo, cuerpo y alrededores. **No es una detección más grande ni una prueba de que se localizó todo el cuerpo.**

Si el recorte es minúsculo, no se fuerza la clasificación. Si una imagen devuelve «blank», significa **sin especie identificada en ese recorte**. La concordancia entre recortes es informativa, pero tampoco constituye validación científica.

La carpeta temporal corregida evita que SpeciesNet intente escribir sus pesos en la ruta de KaggleHub de solo lectura.

In [ ]:
from speciesnet.classifier import SpeciesNetClassifier
import kagglehub
import json
from pathlib import Path
from shutil import copy2

species_suggestions=[]
species_status="not_run"
species_note=""
species_views=[]
exact_crop=None
context_crop=None

def prepare_classifier_model():
    official_id="google/speciesnet/pyTorch/v4.0.3a/1"
    original=Path(kagglehub.model_download(official_id)).resolve()
    metadata=json.loads((original/"info.json").read_text(encoding="utf-8"))
    target=folder/"speciesnet_classifier_only"
    target.mkdir(parents=True,exist_ok=True)
    for key in ("classifier","classifier_labels"):
        if key not in metadata:
            raise RuntimeError("El modelo oficial no contiene "+key)
        relative=Path(metadata[key])
        if relative.is_absolute() or ".." in relative.parts:
            raise RuntimeError("Ruta de pesos inesperada: "+str(relative))
        source_file=original/relative
        if not source_file.is_file():
            raise FileNotFoundError(str(source_file))
        target_file=target/relative
        target_file.parent.mkdir(parents=True,exist_ok=True)
        if not target_file.exists():
            try:
                target_file.symlink_to(source_file)
            except OSError:
                copy2(source_file,target_file)
    metadata["detector"]="unused_in_classifier_only_test.pt"
    (target/"info.json").write_text(json.dumps(metadata,indent=2),encoding="utf-8")
    return target

if people:
    species_status="person_detected"
    species_note="Se detectó una persona. Se omite clasificación de especies por privacidad."
elif primary_detection is None:
    species_status="no_animal"
    species_note="MegaDetector no encontró animales."
elif primary_detection["box_fraction_of_image"]<0.01:
    species_status="insufficient_bbox"
    species_note="Caja principal demasiado pequeña para esta prueba de clasificación."
else:
    x1,y1,x2,y2=primary_detection["bbox_xyxy"]
    exact_crop=photo.crop((x1,y1,x2,y2))
    w,h=x2-x1,y2-y1
    margin=0.65
    padded_bbox=(max(0,int(x1-w*margin)),max(0,int(y1-h*margin)),
                 min(W,int(x2+w*margin)),min(H,int(y2+h*margin)))
    context_crop=photo.crop(padded_bbox)

    try:
        print("Cargando clasificador SpeciesNet…")
        local_model_dir=prepare_classifier_model()
        classifier=SpeciesNetClassifier(model_name=str(local_model_dir),device=device)
        for crop_name,crop_image in [("recuadro_del_detector",exact_crop),
                                     ("recorte_con_contexto",context_crop)]:
            result=classifier.predict(crop_name+".jpg",classifier.preprocess(crop_image))
            classifications=result.get("classifications",{})
            suggestions=[{
                "label":str(label).rsplit(";",1)[-1].strip(),
                "raw_label":str(label),
                "score":round(float(score),4)
            } for label,score in zip(
                classifications.get("classes",[]),
                classifications.get("scores",[])
            )][:5]
            top=suggestions[0]["label"].lower() if suggestions else ""
            classification="not_identified" if top=="blank" else (
                "suggested" if suggestions else "no_prediction")
            species_views.append({
                "crop":crop_name,
                "bbox_xyxy":list(primary_detection["bbox_xyxy"]) if crop_name=="recuadro_del_detector" else list(padded_bbox),
                "model_generated_bbox":crop_name=="recuadro_del_detector",
                "status":classification,
                "suggestions":suggestions
            })
            print("\n"+crop_name.replace("_"," ").upper())
            for suggestion in suggestions[:3]:
                print("  ",suggestion["label"],round(suggestion["score"]*100,2),"%",
                      "— sugerencia sin validar")
        valid=[view for view in species_views if view["status"]=="suggested"]
        if valid:
            species_status="suggested"
            # El recorte exacto es el predeterminado; el ampliado solo añade contexto.
            preferred=next((view for view in valid if view["crop"]=="recuadro_del_detector"),valid[0])
            species_suggestions=preferred["suggestions"]
            species_note="Sugerencias reales; comparar las dos vistas y confirmar manualmente."
        else:
            species_status="not_identified"
            species_note="Ningún recorte produjo una especie reconocible."
    except Exception as exc:
        species_status="error"
        species_note=type(exc).__name__+": "+str(exc)[:230]
        print("No se pudo finalizar SpeciesNet:",species_note)

print("\nESTADO:",species_status,"·",species_note)

# Presentación explícita: foto original COMPLETA + detección real + recortes.
from PIL import ImageFont
panels=[("FOTO COMPLETA · original",photo),
        ("DETECCIONES REALES · cajas del modelo",annotated)]
if exact_crop is not None:
    panels.append(("RECORTE EXACTO · de MegaDetector",exact_crop))
if context_crop is not None:
    panels.append(("CONTEXTO EXTRA · NO es detección",context_crop))
panel_width=640
panel_pics=[]
for title,img in panels:
    preview=img.copy()
    preview.thumbnail((panel_width,660))
    background=Image.new("RGB",(panel_width,preview.height+46),"#101e2b")
    overlay=ImageDraw.Draw(background)
    overlay.text((12,14),title,fill="white")
    background.paste(preview,((panel_width-preview.width)//2,40))
    panel_pics.append(background)
comparison=Image.new("RGB",(panel_width,sum(i.height for i in panel_pics)+12*(len(panel_pics)-1)),"#102636")
offset=0
for image in panel_pics:
    comparison.paste(image,(0,offset))
    offset+=image.height+12
comparison.save(folder/"comparacion_visual_pantheralab.jpg",quality=90)
display(comparison)
print("Imagen comparativa guardada. El recorte con contexto es una ampliación experimental, no una caja del modelo.")

## 5. Informe y evidencia visual

Se descargan: 1) JSON con **cada detección, orientación y confianza**, 2) imagen anotada con **cajas reales**, y 3) comparación visual de **foto completa, detección y recortes**. No se envían automáticamente a PantheraLIVE ni se consideran observaciones científicas verificadas.

In [ ]:
from datetime import datetime, timezone
import json
summary={
    "platform":"PantheraLAB",
    "test":"Colab MegaDetector V6 Extra multi-orientation + SpeciesNet dual-view",
    "timestamp":datetime.now(timezone.utc).isoformat(),
    "simulation":False,
    "scientifically_verified":False,
    "source":"authorized_image_uploaded_to_colab",
    "image_dimensions":{"width":W,"height":H},
    "megadetector":{
        "model":variant,"threshold":threshold,"orientations_tested":orientations,
        "detections":detections,
        "primary_detection":primary_detection,
        "primary_box_selection":"largest among animal boxes within 0.15 of max confidence; experimental"
    },
    "speciesnet":{
        "status":species_status,"note":species_note,
        "suggestions":species_suggestions,"views":species_views,
        "context_bbox_is_model_detection":False
    },
    "saved_in_pantheralive":False
}
report_file=folder/"resultado_pantheralab.json"
report_file.write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding="utf-8")
print("RESULTADO:")
print(json.dumps(summary,ensure_ascii=False,indent=2)[:12000])
print("Estas probabilidades son puntuaciones de modelos, no identificaciones validadas.")

In [ ]:
from google.colab import files
files.download(str(report_file))
files.download(str(folder/"detecciones.jpg"))
files.download(str(folder/"comparacion_visual_pantheralab.jpg"))

**Límite científico:** las rotaciones y el margen contextual son herramientas experimentales de evaluación de imágenes, no un algoritmo validado de segmentación del cuerpo. La detección automática en video real sigue pendiente de un servidor dedicado con modelos instalados, autenticación, almacenamiento seguro y evaluación con fotos de cámaras trampa.

**Siguiente paso:** comparar ambos resultados con la misma foto de tu perro y comprobar si hay alguna caja real que cubra más cuerpo. Si ninguna orientación lo consigue, no debe afirmarse que el cuerpo completo fue detectado.